In [1]:
import json
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
IMAGE_PATH = "forest_and_roads.tif"
JSON_PATH = "forest_and_roads.json"
PATCH_SIZE = 64
OUTPUT_DIR = "dataset"

img = cv2.imread(IMAGE_PATH, cv2.IMREAD_COLOR)
if img is None:
    raise RuntimeError(f"Cannot load {IMAGE_PATH}")

with open(JSON_PATH, 'r') as f:
    data = json.load(f)

samples = []

for shape in data['shapes']:
    label = shape['label'].lower()
    points = shape['points']
    x_center = int((points[0][0] + points[1][0]) / 2)
    y_center = int((points[0][1] + points[1][1]) / 2)
    samples.append((x_center, y_center, label))

print(f"Found {len(samples)} rectangles")

classes = list(set([s[2] for s in samples]))
for cls in classes:
    os.makedirs(os.path.join(OUTPUT_DIR, 'train', cls), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, 'val', cls), exist_ok=True)

train_samples, val_samples = train_test_split(samples, test_size=0.2, random_state=42)

def extract_patch(img, cx, cy, size):
    half = size // 2
    x1 = cx - half
    y1 = cy - half
    x2 = cx + half
    y2 = cy + half
    h, w = img.shape[:2]
    
    pad_top = max(0, -y1)
    pad_bottom = max(0, y2 - h)
    pad_left = max(0, -x1)
    pad_right = max(0, x2 - w)
    
    if pad_top > 0 or pad_bottom > 0 or pad_left > 0 or pad_right > 0:
        img_padded = cv2.copyMakeBorder(img, pad_top, pad_bottom, pad_left, pad_right,
                                        cv2.BORDER_REFLECT)
        cx_pad = cx + pad_left
        cy_pad = cy + pad_top
        x1_pad = cx_pad - half
        y1_pad = cy_pad - half
        x2_pad = cx_pad + half
        y2_pad = cy_pad + half
        patch = img_padded[y1_pad:y2_pad, x1_pad:x2_pad]
    else:
        patch = img[y1:y2, x1:x2]
    return patch

def save_samples(sample_list, train_or_val):
    for i, (cx, cy, cls) in enumerate(sample_list):
        patch = extract_patch(img, cx, cy, PATCH_SIZE)
        out_path = os.path.join(OUTPUT_DIR, train_or_val, cls, f"{i}.png")
        cv2.imwrite(out_path, patch)
    print(f"Saved {len(sample_list)} patches to {train_or_val}")

save_samples(train_samples, 'train')
save_samples(val_samples, 'val')

Found 75 rectangles
Saved 60 patches to train
Saved 15 patches to val
